In [1]:
import json
import pandas as pd
import re

conversations = []
with open("../data/processed/amazon_conversations_clean.jsonl", encoding="utf-8") as f:
    for line in f:
        conversations.append(json.loads(line))

first_customer_msgs = []
for conv in conversations:
    first_turn = next((t for t in conv["turns"] if t["role"] == "CUSTOMER"), None)
    if first_turn and len(first_turn["cleaned_text"].strip()) > 0:
        first_customer_msgs.append({
            "conversation_id": conv["conversation_id"],
            "text": first_turn["cleaned_text"],
            "num_turns": len(conv["turns"])
        })

df_msgs = pd.DataFrame(first_customer_msgs)
print("Total messages:", len(df_msgs))

Total messages: 60669


In [2]:
taxonomy_patterns = {
    "delivery_status": [r"\bdeliver", r"\bpackage\b", r"\btrack", r"\barriv", r"\bshipp", r"\bwhere is\b", r"\breceived?\b", r"\brcvd\b"],
    "order_issue": [r"\border\b", r"\bordered\b", r"\bcancel", r"\bwrong item\b"],
    "refund_return": [r"\brefund", r"\breturn", r"\bmoney back\b", r"\bexchange\b", r"\bcashback\b"],
    "payment_billing": [r"\bcharge", r"\bpayment\b", r"\bbill", r"\bcard\b"],
    "pricing_query": [r"\bmrp\b", r"\bprice\b", r"\bpricing\b", r"\boverchar", r"\bcost\b"],
    "prime_membership": [r"\bprime\b", r"\bmembership\b", r"\bsubscription\b"],
    "content_streaming_issue": [r"\bprime video\b", r"\baudible\b", r"\bsubtitle", r"\bepisode\b", r"\bgeo setting", r"\bregion\b", r"\bstream"],
    "device_tech_support": [r"\becho\b", r"\balexa\b", r"\bkindle\b", r"\bwifi\b", r"\bapp\b", r"\bdevice\b", r"\bconnect", r"\bupdate\b"],
    "repair_service_status": [r"\brepair\b", r"\bservice cent", r"\bsubmitted\b.*\b(mobile|device|phone)\b"],
    "account_access": [r"\baccount\b", r"\bpassword\b", r"\blogin\b", r"\block", r"\baccess\b"],
    "availability_question": [r"\bwhen (will|would)\b", r"\bavailable\b", r"\bavailability\b"],
    "promo_discount_query": [r"\bpromo code\b", r"\bdiscount\b", r"\bcoupon\b", r"\boffer\b"],
    "feature_request": [r"\bwould be (cool|nice|great)\b", r"\bit'?d be (cool|nice)\b", r"\bsuggestion\b", r"\bfeature\b"],
    "product_issue": [r"\bdamaged\b", r"\bbroken\b", r"\bdefective\b", r"\bnot working\b", r"\bfaulty\b"],
    "customer_service_complaint": [r"\bcustomer service\b", r"\brude\b", r"\bworst\b", r"\bterrible\b", r"\bdisappointed\b", r"\bhorrible\b"],
    "human_assistance_request": [r"\bhuman\b", r"\breal person\b", r"\bspeak to\b", r"\bcall me\b", r"\brepresentative\b"],
}

def get_matching_categories(text, patterns_dict):
    text_low = text.lower()
    matches = []
    for name, patterns in patterns_dict.items():
        if any(re.search(p, text_low) for p in patterns):
            matches.append(name)
    return matches

df_msgs["candidate_labels"] = df_msgs["text"].apply(lambda t: get_matching_categories(t, taxonomy_patterns))
df_msgs["primary_label"] = df_msgs["candidate_labels"].apply(lambda x: x[0] if x else "UNKNOWN")

print(df_msgs["primary_label"].value_counts())

primary_label
delivery_status               24697
UNKNOWN                       14153
order_issue                    6973
device_tech_support            2967
refund_return                  2481
prime_membership               2419
customer_service_complaint     1682
payment_billing                1551
account_access                 1522
pricing_query                   525
product_issue                   452
content_streaming_issue         392
availability_question           391
promo_discount_query            204
human_assistance_request        146
feature_request                  81
repair_service_status            33
Name: count, dtype: int64


In [3]:
import random
random.seed(42)

target_counts = {
    "delivery_status": 20, "order_issue": 20, "prime_membership": 18,
    "refund_return": 15, "payment_billing": 15, "customer_service_complaint": 15, "account_access": 15,
    "pricing_query": 8, "content_streaming_issue": 8, "device_tech_support": 10,
    "repair_service_status": 8, "availability_question": 8, "promo_discount_query": 8,
    "feature_request": 8, "product_issue": 10, "human_assistance_request": 10,
    "UNKNOWN": 15,
}

sampled_frames = []
for label, n in target_counts.items():
    subset = df_msgs[df_msgs["primary_label"] == label]
    n_take = min(n, len(subset))
    sampled_frames.append(subset.sample(n_take, random_state=42))
    print(f"{label}: requested {n}, available {len(subset)}, taken {n_take}")

golden_candidates = pd.concat(sampled_frames).reset_index(drop=True)
print(f"\nTotal sampled: {len(golden_candidates)}")

delivery_status: requested 20, available 24697, taken 20
order_issue: requested 20, available 6973, taken 20
prime_membership: requested 18, available 2419, taken 18
refund_return: requested 15, available 2481, taken 15
payment_billing: requested 15, available 1551, taken 15
customer_service_complaint: requested 15, available 1682, taken 15
account_access: requested 15, available 1522, taken 15
pricing_query: requested 8, available 525, taken 8
content_streaming_issue: requested 8, available 392, taken 8
device_tech_support: requested 10, available 2967, taken 10
repair_service_status: requested 8, available 33, taken 8
availability_question: requested 8, available 391, taken 8
promo_discount_query: requested 8, available 204, taken 8
feature_request: requested 8, available 81, taken 8
product_issue: requested 10, available 452, taken 10
human_assistance_request: requested 10, available 146, taken 10
UNKNOWN: requested 15, available 14153, taken 15

Total sampled: 211


In [4]:
import uuid

golden_set = []
for i, row in golden_candidates.iterrows():
    example_id = f"GS-{i+1:04d}"
    conv = next(c for c in conversations if c["conversation_id"] == row["conversation_id"])
    
    golden_set.append({
        "example_id": example_id,
        "brand_id": "AmazonHelp",
        "conversation_id": row["conversation_id"],
        "current_message": row["text"],
        "context": conv["turns"],  # poori conversation, reference ke liye
        "candidate_label": row["primary_label"],  # regex se suggest hua, FINAL NAHI
        "gold_intent": None,          # <-- manual labeling me fill karenge
        "ambiguity_flag": None,       # <-- manual labeling me fill karenge
        "gold_escalation": None,      # <-- manual labeling me fill karenge (AUTO_HANDLE/ESCALATE)
        "expected_behavior": None,    # <-- manual labeling me fill karenge
        "evidence_relevance": None,   # optional, baad me
        "notes": "",
    })

with open("../data/schemas/golden_set_draft.jsonl", "w", encoding="utf-8") as f:
    for ex in golden_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Saved {len(golden_set)} draft examples to data/schemas/golden_set_draft.jsonl")

Saved 211 draft examples to data/schemas/golden_set_draft.jsonl


In [5]:
label_rows = []
for ex in golden_set:
    label_rows.append({
        "example_id": ex["example_id"],
        "current_message": ex["current_message"],
        "candidate_label (suggested)": ex["candidate_label"],
        "gold_intent (YOU FILL)": "",
        "ambiguity_flag (yes/no)": "",
        "gold_escalation (AUTO_HANDLE/ESCALATE)": "",
        "notes": "",
    })

label_df = pd.DataFrame(label_rows)
label_df.to_csv("../data/schemas/golden_set_labeling_sheet.csv", index=False)
print("Saved labeling sheet to data/schemas/golden_set_labeling_sheet.csv")

Saved labeling sheet to data/schemas/golden_set_labeling_sheet.csv


In [6]:
for i, ex in enumerate(golden_set[:20]):
    print(f"\n{'='*60}")
    print(f"[{ex['example_id']}] (regex suggested: {ex['candidate_label']})")
    print(f"Message: {ex['current_message']}")
    print(f"Full context ({len(ex['context'])} turns):")
    for t in ex['context'][:4]:  # pehle 4 turns dikhado, zyada lamba na ho
        print(f"   [{t['role']}] {t['cleaned_text'][:150]}")


[GS-0001] (regex suggested: delivery_status)
Message: its says that my product has delivere but i haven’t received it yet
Full context (6 turns):
   [CUSTOMER] its says that my product has delivere but i haven’t received it yet
   [BRAND] I'm sorry for this wait. Do you have a Safe Space designated for your deliveries?
   [CUSTOMER] No
   [BRAND] Setting a Safe Place on the account can be done by following these helpful steps:

[GS-0002] (regex suggested: delivery_status)
Message: its been out for delivery since 9am BLS I need it fuck this fuck amazon prime 😢😢😢 why is every damn package delivery a nightmare these days
Full context (6 turns):
   [CUSTOMER] its been out for delivery since 9am BLS I need it fuck this fuck amazon prime 😢😢😢 why is every damn package delivery a nightmare these days
   [BRAND] I'm so sorry for your frustration! Packages have until 8PM to be delivered. Please let us know if you have any further issues!
   [CUSTOMER] do these times apply to german amazon logis

In [7]:
with open("../data/schemas/labeling_batch_1.txt", "w", encoding="utf-8") as f:
    for i, ex in enumerate(golden_set[:20]):
        f.write(f"\n{'='*60}\n")
        f.write(f"[{ex['example_id']}] (regex suggested: {ex['candidate_label']})\n")
        f.write(f"Message: {ex['current_message']}\n")
        for t in ex['context'][:4]:
            f.write(f"   [{t['role']}] {t['cleaned_text'][:150]}\n")

print("Saved. Ab is file ko VS Code me khol ke text copy-paste kar de mujhe.")

Saved. Ab is file ko VS Code me khol ke text copy-paste kar de mujhe.


In [8]:
batch_1_labels = {
    "GS-0001": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0002": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0003": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0004": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0005": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0006": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0007": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0008": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0009": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0010": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0011": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0012": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0013": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0014": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0015": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0016": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0017": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0018": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0019": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "not a genuine support request — positive/off-topic tweet"},
    "GS-0020": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
}

for ex in golden_set:
    if ex["example_id"] in batch_1_labels:
        ex.update(batch_1_labels[ex["example_id"]])

labeled_count = sum(1 for ex in golden_set if ex["gold_intent"] is not None)
print(f"Labeled so far: {labeled_count}/{len(golden_set)}")

# Progress ko save kar do (resume karne ke liye, kabhi bhi notebook restart ho)
with open("../data/schemas/golden_set_progress.jsonl", "w", encoding="utf-8") as f:
    for ex in golden_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print("Progress saved to golden_set_progress.jsonl")

Labeled so far: 20/211
Progress saved to golden_set_progress.jsonl


In [9]:
with open("../data/schemas/labeling_batch_2.txt", "w", encoding="utf-8") as f:
    for i, ex in enumerate(golden_set[20:40]):
        f.write(f"\n{'='*60}\n")
        f.write(f"[{ex['example_id']}] (regex suggested: {ex['candidate_label']})\n")
        f.write(f"Message: {ex['current_message']}\n")
        for t in ex['context'][:4]:
            f.write(f"   [{t['role']}] {t['cleaned_text'][:150]}\n")

print("Saved batch 2 to labeling_batch_2.txt — VS Code me khol ke text paste kar.")

Saved batch 2 to labeling_batch_2.txt — VS Code me khol ke text paste kar.


In [10]:
batch_2_labels = {
    "GS-0021": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0022": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0023": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0024": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0025": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0026": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0027": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0028": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0029": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0030": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0031": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0032": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0033": {"gold_intent": "packaging_feedback", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0034": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0035": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0036": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0037": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0038": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0039": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0040": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "fragment message, unclear underlying issue"},
}

for ex in golden_set:
    if ex["example_id"] in batch_2_labels:
        ex.update(batch_2_labels[ex["example_id"]])

labeled_count = sum(1 for ex in golden_set if ex["gold_intent"] is not None)
print(f"Labeled so far: {labeled_count}/{len(golden_set)}")

with open("../data/schemas/golden_set_progress.jsonl", "w", encoding="utf-8") as f:
    for ex in golden_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print("Progress saved.")

Labeled so far: 40/211
Progress saved.


In [11]:
with open("../data/schemas/labeling_batch_3.txt", "w", encoding="utf-8") as f:
    for i, ex in enumerate(golden_set[40:60]):
        f.write(f"\n{'='*60}\n")
        f.write(f"[{ex['example_id']}] (regex suggested: {ex['candidate_label']})\n")
        f.write(f"Message: {ex['current_message']}\n")
        for t in ex['context'][:4]:
            f.write(f"   [{t['role']}] {t['cleaned_text'][:150]}\n")

print("Saved batch 3 to labeling_batch_3.txt — VS Code me khol ke text paste kar.")

Saved batch 3 to labeling_batch_3.txt — VS Code me khol ke text paste kar.


In [12]:
batch_3_labels = {
    "GS-0041": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0042": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "vague message, real issue only clear from brand's follow-up"},
    "GS-0043": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "vague but escalating/concerning tone"},
    "GS-0044": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0045": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0046": {"gold_intent": "UNKNOWN", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0047": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0048": {"gold_intent": "promo_discount_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0049": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0050": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0051": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "not a genuine support request — positive tweet"},
    "GS-0052": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0053": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0054": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0055": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0056": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0057": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0058": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0059": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0060": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
}

for ex in golden_set:
    if ex["example_id"] in batch_3_labels:
        ex.update(batch_3_labels[ex["example_id"]])

labeled_count = sum(1 for ex in golden_set if ex["gold_intent"] is not None)
print(f"Labeled so far: {labeled_count}/{len(golden_set)}")

with open("../data/schemas/golden_set_progress.jsonl", "w", encoding="utf-8") as f:
    for ex in golden_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print("Progress saved.")

Labeled so far: 60/211
Progress saved.


In [13]:
with open("../data/schemas/labeling_remaining.txt", "w", encoding="utf-8") as f:
    for ex in golden_set[60:]:  # GS-0061 se aakhir tak
        f.write(f"\n{'='*60}\n")
        f.write(f"[{ex['example_id']}] (regex suggested: {ex['candidate_label']})\n")
        f.write(f"Message: {ex['current_message']}\n")
        for t in ex['context'][:4]:
            f.write(f"   [{t['role']}] {t['cleaned_text'][:150]}\n")

print(f"Saved {len(golden_set) - 60} remaining examples to labeling_remaining.txt")

Saved 151 remaining examples to labeling_remaining.txt


In [14]:
remaining_labels = {
    "GS-0061": {"gold_intent": "human_assistance_request", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0062": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0063": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0064": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0065": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0066": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0067": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0068": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0069": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0070": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0071": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0072": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0073": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0074": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0075": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0076": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0077": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0078": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0079": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0080": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0081": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0082": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0083": {"gold_intent": "pricing_query", "ambiguity_flag": "yes", "gold_escalation": "AUTO_HANDLE"},
    "GS-0084": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0085": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0086": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0087": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0088": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0089": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0090": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE"},
    "GS-0091": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "yes", "gold_escalation": "AUTO_HANDLE"},
    "GS-0092": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0093": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0094": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0095": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE"},
    "GS-0096": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0097": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0098": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE"},
    "GS-0099": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0100": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0101": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0102": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive/off-topic tweet, not a support request"},
    "GS-0103": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0104": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0105": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0106": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0107": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0108": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0109": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0110": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0111": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0112": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0113": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0114": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0115": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0116": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0117": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0118": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0119": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0120": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0121": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0122": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0123": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0124": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0125": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0126": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0127": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0128": {"gold_intent": "device_tech_support", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0129": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0130": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0131": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0132": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0133": {"gold_intent": "device_tech_support", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0134": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0135": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0136": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0137": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0138": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0139": {"gold_intent": "device_tech_support", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0140": {"gold_intent": "device_tech_support", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0141": {"gold_intent": "device_tech_support", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0142": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0143": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0144": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0145": {"gold_intent": "repair_service_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0146": {"gold_intent": "repair_service_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0147": {"gold_intent": "repair_service_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0148": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0149": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0150": {"gold_intent": "repair_service_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0151": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0152": {"gold_intent": "repair_service_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0153": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0154": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0155": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0156": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0157": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0158": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0159": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0160": {"gold_intent": "availability_question", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0161": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "sarcastic, unclear real request"},
    "GS-0162": {"gold_intent": "promo_discount_query", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0163": {"gold_intent": "promo_discount_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0164": {"gold_intent": "promo_discount_query", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0165": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0166": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE"},
    "GS-0167": {"gold_intent": "promo_discount_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0168": {"gold_intent": "promo_discount_query", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0169": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0170": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0171": {"gold_intent": "content_streaming_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0172": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0173": {"gold_intent": "human_assistance_request", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0174": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0175": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0176": {"gold_intent": "feature_request", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0177": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0178": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0179": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0180": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0181": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0182": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0183": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0184": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0185": {"gold_intent": "repair_service_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0186": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0187": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0188": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0189": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE"},
    "GS-0190": {"gold_intent": "refund_return", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0191": {"gold_intent": "human_assistance_request", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0192": {"gold_intent": "human_assistance_request", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0193": {"gold_intent": "human_assistance_request", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0194": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0195": {"gold_intent": "human_assistance_request", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0196": {"gold_intent": "delivery_status", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0197": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0198": {"gold_intent": "payment_billing", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0199": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "scam warning, not a personal support need"},
    "GS-0200": {"gold_intent": "account_access", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0201": {"gold_intent": "promo_discount_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0202": {"gold_intent": "prime_membership", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0203": {"gold_intent": "human_assistance_request", "ambiguity_flag": "yes", "gold_escalation": "AUTO_HANDLE"},
    "GS-0204": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0205": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE", "notes": "positive tweet, not a support request"},
    "GS-0206": {"gold_intent": "product_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
    "GS-0207": {"gold_intent": "customer_service_complaint", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0208": {"gold_intent": "packaging_feedback", "ambiguity_flag": "yes", "gold_escalation": "AUTO_HANDLE"},
    "GS-0209": {"gold_intent": "pricing_query", "ambiguity_flag": "no", "gold_escalation": "AUTO_HANDLE"},
    "GS-0210": {"gold_intent": "UNKNOWN", "ambiguity_flag": "yes", "gold_escalation": "ESCALATE"},
    "GS-0211": {"gold_intent": "order_issue", "ambiguity_flag": "no", "gold_escalation": "ESCALATE"},
}

for ex in golden_set:
    if ex["example_id"] in remaining_labels:
        ex.update(remaining_labels[ex["example_id"]])

labeled_count = sum(1 for ex in golden_set if ex["gold_intent"] is not None)
print(f"Labeled so far: {labeled_count}/{len(golden_set)}")

with open("../data/schemas/golden_set_progress.jsonl", "w", encoding="utf-8") as f:
    for ex in golden_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print("Progress saved.")

Labeled so far: 211/211
Progress saved.


In [15]:
import hashlib
from datetime import datetime

# Final file me save karo (ye ab "frozen" version hai)
final_path = "../data/schemas/golden_set_final.jsonl"
with open(final_path, "w", encoding="utf-8") as f:
    for ex in golden_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# Checksum nikaalo (reproducibility/version-tracking ke liye)
with open(final_path, "rb") as f:
    content = f.read()
    checksum = hashlib.sha256(content).hexdigest()

version_info = {
    "version": "golden_set_v1",
    "frozen_at": datetime.now().isoformat(),
    "total_examples": len(golden_set),
    "sha256_checksum": checksum,
    "brand_id": "AmazonHelp",
    "taxonomy_version": "intent_taxonomy_v1",
    "sampling_method": "stratified by regex-proxy candidate label, manually reviewed and corrected",
}

with open("../data/schemas/golden_set_version.json", "w", encoding="utf-8") as f:
    json.dump(version_info, f, indent=2)

print("Frozen golden set saved with version info:")
print(json.dumps(version_info, indent=2))

Frozen golden set saved with version info:
{
  "version": "golden_set_v1",
  "frozen_at": "2026-09-11T10:15:01.620713",
  "total_examples": 211,
  "sha256_checksum": "8ea6e6463b1a76820e47b75af9f623be058816f1b0a8c663e7d19267fd9d9af9",
  "brand_id": "AmazonHelp",
  "taxonomy_version": "intent_taxonomy_v1",
  "sampling_method": "stratified by regex-proxy candidate label, manually reviewed and corrected"
}


In [16]:
import pandas as pd

df_gold = pd.DataFrame(golden_set)

print("=== Gold Intent Distribution ===")
print(df_gold["gold_intent"].value_counts())

print("\n=== Escalation Distribution ===")
print(df_gold["gold_escalation"].value_counts())
print(f"\nEscalation rate: {100 * (df_gold['gold_escalation']=='ESCALATE').mean():.1f}%")

print("\n=== Ambiguity Distribution ===")
print(df_gold["ambiguity_flag"].value_counts())

print("\n=== Regex-suggestion vs Final-label agreement ===")
df_gold["regex_correct"] = df_gold["candidate_label"] == df_gold["gold_intent"]
print(f"Regex proxy matched final label: {df_gold['regex_correct'].mean()*100:.1f}% of the time")

=== Gold Intent Distribution ===
gold_intent
payment_billing               26
delivery_status               25
UNKNOWN                       18
product_issue                 17
customer_service_complaint    14
content_streaming_issue       13
account_access                13
refund_return                 12
availability_question         12
order_issue                   12
pricing_query                 11
feature_request               10
promo_discount_query           7
human_assistance_request       7
repair_service_status          6
device_tech_support            5
packaging_feedback             2
prime_membership               1
Name: count, dtype: int64

=== Escalation Distribution ===
gold_escalation
ESCALATE       109
AUTO_HANDLE    102
Name: count, dtype: int64

Escalation rate: 51.7%

=== Ambiguity Distribution ===
ambiguity_flag
no     185
yes     26
Name: count, dtype: int64

=== Regex-suggestion vs Final-label agreement ===
Regex proxy matched final label: 50.2% of the time
